In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# Caminho para a pasta onde os dados foram extraídos
DATASET_DIR = "../data/dataset_hibrida" 

BATCH_SIZE = 32
# A MobileNetV2 espera imagens quadradas de 224x224
IMG_SIZE = (224, 224) 

print("Carregando conjunto de TREINO:")
treino_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "treino"),
    shuffle=True, # Embaralha para o modelo não viciar na ordem
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
)

print("\nCarregando conjunto de VALIDAÇÃO:")
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "validacao"),
    shuffle=False, # Não precisamos embaralhar validação
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
)

print("\nCarregando conjunto de TESTE:")
teste_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "teste"),
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
)

class_names = treino_ds.class_names
print(f"\nClasses detectadas: {class_names}")

# Otimização de I/O focada em poupar memória RAM
AUTOTUNE = tf.data.AUTOTUNE
# O .cache() foi removido. O prefetch() já é suficiente para manter o fluxo contínuo!
treino_ds = treino_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
teste_ds = teste_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# 1. Camada de Aumento de Dados (Data Augmentation)
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
    ],
    name="data_augmentation",
)

# 2. Carregando a MobileNetV2 pré-treinada
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, 
    weights='imagenet'
)

# Congelando os pesos da base
base_model.trainable = False

# 3. Montando o modelo final
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)

# Normalização de pixels (-1 a 1) exigida pela MobileNetV2
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
x = preprocess_input(x)

# Passando as imagens pela base congelada
x = base_model(x, training=False)

# Achata as características encontradas
x = layers.GlobalAveragePooling2D()(x)

# Dropout de 20% para evitar overfitting
x = layers.Dropout(0.2)(x) 

# ALTERAÇÃO 1: Camada de Saída com 3 neurônios e ativação 'softmax'
# O Softmax transforma a saída em probabilidades que somam 100% entre as 3 classes
outputs = layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)

# ALTERAÇÃO 2: Compilando o modelo com a função de erro para múltiplas classes
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(), # Substitui a BinaryCrossentropy
    metrics=['accuracy']
)

# Exibe o resumo da arquitetura
model.summary()

In [ ]:
# Callbacks para otimizar o treinamento
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', # Fica de olho no erro da validação
        patience=5,         # Se não melhorar por 5 épocas seguidas, ele para
        restore_best_weights=True # Garante que vai salvar a melhor versão, não a última
    )
]

EPOCHS = 20 # Número máximo de vezes que ele vai ver o dataset inteiro

print("Iniciando o treinamento da Baseline de Doenças...")
history = model.fit(
    treino_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Plotando os gráficos de aprendizado
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia no Treino')
plt.plot(epochs_range, val_acc, label='Acurácia na Validação')
plt.legend(loc='lower right')
plt.title('Acurácia')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Erro no Treino (Loss)')
plt.plot(epochs_range, val_loss, label='Erro na Validação')
plt.legend(loc='upper right')
plt.title('Erro (Loss)')
plt.show()

In [ ]:
# Salvando o modelo treinado localmente
caminho_modelo = "baseline_pinta_preta.keras"
model.save(caminho_modelo)
print(f"Modelo salvo com sucesso em: {caminho_modelo}")

# Caso precise carregar depois sem ter que treinar tudo de novo:
# model = keras.models.load_model(caminho_modelo)

In [ ]:
print("\nAvaliando modelo no Conjunto de Teste (Dados Inéditos)...")
test_loss, test_acc = model.evaluate(teste_ds)
print(f"Acurácia real no Teste: {test_acc:.4f}")
print(f"Erro (Loss) no Teste: {test_loss:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Extraindo rótulos reais e previsões lote por lote para poupar RAM
y_true = []
y_pred_probs = []

for images, labels in teste_ds:
    y_true.extend(labels.numpy())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
# CORREÇÃO MULTICLASSE: Pega o índice da maior probabilidade entre as 3 opções
y_pred = np.argmax(y_pred_probs, axis=1)

# CORREÇÃO DAS CLASSES: Ordem alfabética exata conforme log da Célula 1
classes = ['distrator_septoria', 'pinta_preta', 'saudavel']

print("\n--- Relatório de Classificação ---")
print(classification_report(y_true, y_pred, target_names=classes))

# Plotando a Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusão - Dataset Híbrido')
plt.ylabel('Rótulo Verdadeiro (Real)')
plt.xlabel('Previsão do Modelo')
plt.show()

In [ ]:
# Puxa o primeiro lote (batch) do conjunto de teste
plt.figure(figsize=(15, 10))
for images, labels in teste_ds.take(1):
    preds = model.predict(images, verbose=0)
    
    for i in range(12): # Vamos olhar 12 folhas
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        # Pega o índice real e o previsto (aquele com a maior probabilidade)
        indice_real = int(labels[i])
        indice_previsto = np.argmax(preds[i])
        
        classe_real = classes[indice_real]
        classe_prevista = classes[indice_previsto]
        
        # A confiança é o valor da maior probabilidade calculada pelo Softmax
        confianca = np.max(preds[i])
        
        cor_titulo = "green" if classe_real == classe_prevista else "red"
        
        plt.title(f"Previsão: {classe_prevista}\nReal: {classe_real}\nConfiança: {confianca*100:.1f}%", color=cor_titulo)
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

def testar_imagens_da_pasta(diretorio, modelo):
    # Procura por todas as imagens (jpg, jpeg, png) na sua pasta
    extensoes = ('*.jpg', '*.jpeg', '*.png')
    arquivos_imagens = []
    
    for ext in extensoes:
        arquivos_imagens.extend(glob.glob(os.path.join(diretorio, ext)))
        # Tenta achar também com letras maiúsculas (ex: .JPG)
        arquivos_imagens.extend(glob.glob(os.path.join(diretorio, ext.upper())))
        
    if not arquivos_imagens:
        print(f"Nenhuma imagem encontrada na pasta:\n{diretorio}\nSalve algumas fotos lá e rode novamente!")
        return

    print(f"Encontradas {len(arquivos_imagens)} imagens para teste. Iniciando diagnósticos...\n")

    # Lista de classes na ordem exata definida pelo modelo
    classes = ['distrator_septoria', 'pinta_preta', 'saudavel']

    for caminho_imagem in arquivos_imagens:
        try:
            # 1. Carregar a imagem
            img = image.load_img(caminho_imagem, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            
            # 2. Fazer a previsão
            previsao = modelo.predict(img_array, verbose=0)
            
            # 3. Interpretar o resultado (MULTICLASSE)
            indice_previsto = np.argmax(previsao[0])
            classe_prevista = classes[indice_previsto]
            confianca = np.max(previsao[0]) * 100
            
            # Formatação visual dependendo da classe
            if classe_prevista == "saudavel":
                cor = "green"
                diagnostico = "Saudável"
            elif classe_prevista == "pinta_preta":
                cor = "red"
                diagnostico = "Pinta-Preta"
            else:
                cor = "orange"
                diagnostico = "Septoria (Distrator)"
                
            # 4. Plotar o resultado
            plt.figure(figsize=(4, 4))
            plt.imshow(img)
            nome_arquivo = os.path.basename(caminho_imagem)
            plt.title(f"Arquivo: {nome_arquivo}\nDiagnóstico: {diagnostico}\nConfiança: {confianca:.1f}%", 
                      color=cor, fontweight='bold')
            plt.axis("off")
            plt.show()
            
        except Exception as e:
            print(f"Erro ao processar a imagem {caminho_imagem}: {e}")

# ==========================================
# EXECUÇÃO DO TESTE
# ==========================================
DIRETORIO_TESTE = r"C:\Users\joaom\OneDrive\Documentos\GitHub\Ecovision\ml\images_test"

# Chama a função passando a sua pasta e o modelo treinado
testar_imagens_da_pasta(DIRETORIO_TESTE, model)

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tensorflow.keras.preprocessing import image

def obter_heatmap_gradcam(img_array, modelo, nome_camada_base):
    # Converte a imagem para Tensor para evitar problemas matemáticos no Keras
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
    
    # 1. Identifica a MobileNetV2 dentro da sua rede
    base_model = modelo.get_layer(nome_camada_base)
    idx_base = modelo.layers.index(base_model)
    
    # 2. Pré-processamento manual seguro (Substitui o loop que causou o erro)
    # A camada de Data Augmentation é ignorada no teste, restando apenas a normalização:
    preprocessed_input = tf.keras.applications.mobilenet_v2.preprocess_input(img_tensor)
    
    # 3. Extrai as características espaciais (Features) da MobileNetV2
    features = base_model(preprocessed_input, training=False)
    
    # 4. Constrói o classificador final isolado (Pooling + Dropout + Dense)
    classifier_input = tf.keras.Input(shape=features.shape[1:])
    x = classifier_input
    for layer in modelo.layers[idx_base + 1:]:
        # Garante que o Dropout não vai desligar neurônios durante o teste
        if isinstance(layer, tf.keras.layers.Dropout):
            x = layer(x, training=False)
        else:
            x = layer(x)
    classifier_model = tf.keras.Model(classifier_input, x)
    
    # 5. Rastreia o pensamento matemático (Gradientes)
    with tf.GradientTape() as tape:
        tape.watch(features) # Comando explícito para a IA focar neste mapa de características
        previsoes = classifier_model(features)
        indice_previsto = tf.argmax(previsoes[0])
        canal_classe = previsoes[:, indice_previsto]
        
    # 6. Calcula a importância de cada pixel para a decisão final
    grads = tape.gradient(canal_classe, features)
    grads_agrupados = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    features = features[0]
    heatmap = features @ grads_agrupados[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # 7. Normaliza o Heatmap de 0 a 1 para virar imagem
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy(), int(indice_previsto), np.max(previsoes[0])

def plotar_prova_gradcam(caminho_imagem, modelo, classes):
    # Busca a camada base dinamicamente pelo nome
    nome_camada_base = next((layer.name for layer in modelo.layers if 'mobilenet' in layer.name.lower()), None)
    
    if not nome_camada_base:
        print("Erro: Camada base MobileNetV2 não encontrada no modelo.")
        return

    # Carrega a imagem do HD
    img = image.load_img(caminho_imagem, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # Gera o Heatmap
    heatmap, idx_previsto, confianca = obter_heatmap_gradcam(img_array, modelo, nome_camada_base)
    classe_prevista = classes[idx_previsto]

    # ==================================================
    # MAPAS DE ATIVAÇÃO (GRAD-CAM)
    # ==================================================
    # 1. Filtro de Limiar: Zera tudo que for menor que 30% de importância (remove o "ruído" do fundo)
    heatmap_limpo = np.where(heatmap > 0.3, heatmap, 0)

    # Transforma os números em cores térmicas (Jet)
    heatmap_redimensionado = np.uint8(255 * heatmap_limpo)
    mapa_cores = plt.get_cmap("jet")
    cores_rgb = mapa_cores(np.arange(256))[:, :3]
    heatmap_colorido = cores_rgb[heatmap_redimensionado]

    # Converte para objeto de imagem do TensorFlow/PIL
    heatmap_img = tf.keras.utils.array_to_img(heatmap_colorido)
    
    # 2. Suavização (Interpolação): Usa o método LANCZOS para borrar as bordas e tirar o quadriculado
    from PIL import Image
    heatmap_img = heatmap_img.resize((img.size[0], img.size[1]), resample=Image.LANCZOS)
    heatmap_array = tf.keras.utils.img_to_array(heatmap_img)

    # ==================================================

    # Cria a imagem final fundida (60% foto original, 40% mapa de calor)
    imagem_original = tf.keras.utils.img_to_array(img)
    imagem_sobreposta = heatmap_array * 0.4 + imagem_original
    imagem_final = tf.keras.utils.array_to_img(imagem_sobreposta)

    # Plotagem da Prova XAI
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img)
    axes[0].set_title("1. Imagem de Entrada")
    axes[0].axis('off')

    # Exibe o mapa isolado já com a interpolação ativada na visualização
    axes[1].imshow(heatmap_limpo, cmap='jet', interpolation='lanczos')
    axes[1].set_title("2. Mapa de Ativação Suavizado")
    axes[1].axis('off')

    axes[2].imshow(imagem_final)
    cor_titulo = "green" if classe_prevista == "saudavel" else "red"
    axes[2].set_title(f"3. Prova de Aprendizado\nPrevisão: {classe_prevista} ({confianca*100:.1f}%)", color=cor_titulo, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# ==========================================
# EXECUÇÃO DA PROVA (GRAD-CAM)
# ==========================================
DIRETORIO_TESTE = r"C:\Users\joaom\OneDrive\Documentos\GitHub\Ecovision\ml\images_test"
extensoes = ('*.jpg', '*.jpeg', '*.png', '*.JPG')
todas_imagens = []
for ext in extensoes:
    todas_imagens.extend(glob.glob(os.path.join(DIRETORIO_TESTE, ext)))

classes = ['distrator_septoria', 'pinta_preta', 'saudavel']

# Varre todas as imagens da sua pasta e gera o raio-x para cada uma
if todas_imagens:
    print(f"Gerando log comprobatório Grad-CAM para {len(todas_imagens)} imagens...\n")
    for caminho in todas_imagens:
        plotar_prova_gradcam(caminho, model, classes)
else:
    print("Nenhuma imagem encontrada na pasta de testes!")